<a href="https://colab.research.google.com/github/AureTrix-Solutions/99_DataShop/blob/main/99.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#<big>**$\color{tan}{\text{Ι. Data Shop Main}}$**</big>
---



##$\color{gray}{\text{A. Browser Support And Code}}$<br>$\color{gray}{\text{Troubleshooting.}}$

---
###<i><dd><dl>$\color{gray}{\text{1. Browser Support}}$</dl></dd></i>

<dd><dl><ins>$\color{Yellow}{\text{Use Microsoft Edge Browser}}$</ins>

Enable 3rd party cookies:

* Go to browser settings (3 dots in the top right corner).
* In the left pane click cookies and site permissions.
* In th middle click manage and delete cookies and site data.
* In the allow section click add.
* copy and paste code below and check the include third party cookies on this site box.
* refresh this page, minimize all the code and then press the play button next to the module you want to use, once the code is loaded you should see drop downs, entry boxes and buttons to interact with the module.
</dd></dl>


```
[*.]google.com
```

###<i><dl><dd>$\color{gray}{\text{2. Code troubleshooting}}$</dd></dl></i>
<dd><dl>

* Press play for the first run, after that you can interact with user interface.
* If you run into any issues or errors try pressing play, that will reload the module and fix most problems
* <strong>If replaying the module doesn't work  run [this code](#scrollTo=SLEHZfgLX-Lj&uniqifier=1)</strong></dd></dl>




##$\color{gray}{\text{B. Quick Links.}}$


---
###<dd><dl><em>$\color{gray}{\text{1. Production hour predictors}}$</dd></dl></em>

<dd><dl>

* [All Bands](#scrollTo=7KGbReo-Op_k&uniqifier=1)</strong><br>

</dd></dl>



###<i><dd><dl>$\color{gray}{\text{2. Manpower and muster tools}}$</dd></dl></i>
<dd><dl>

* [Manpower Throughput](#scrollTo=vhCKwsPPc5Kw&uniqifier=1)</strong><br>
* [Muster Report Categorizer](Future_link)</strong></dd></dl>




###<i><dd><dl>$\color{gray}{\text{3. Cannibalization}}$</dd></dl></i>
<dd><dl>

* [Cannibalization Source List/Count: 910, LBT, UEU](#scrollTo=AFTLx05glDgg&uniqifier=1)</strong>


#<big>**$\color{tan}{\text{II. Production Hours}}$**</big>
---



##$\color{gray}{\text{A. Bench Hours Predictions: All Bands}}$

In [ ]:
# @markdown
import pandas as pd
import joblib
import math
import warnings
import ipywidgets as widgets
from IPython.display import display, clear_output
import numpy as np
from sklearn.preprocessing import LabelEncoder
import os
import zipfile
import requests

class Setup:
    def __init__(self):
        warnings.filterwarnings('ignore')
        self.xmt_type_output = widgets.Output()  # Separate output for dropdown
        self.setup_files()
        self.create_widget()



    def setup_files(self):
        # Determine if the code is running in Google Colab or local runtime
        if os.path.exists(r'C:\Users\ricks\Desktop\colab'):
            # Running in local runtime
            print("Running on local runtime")
            self.model_save_path = r'C:\Users\ricks\Desktop\colab\hours_model'
            self.model_files = 'https://www.dropbox.com/scl/fo/l8fqrr5uolxm6028hnhqj/AHvGdiy_0sHzXg2cejjdwRo?rlkey=wz5ueckmkj12hcv4ca54wls50&st=f3l5la1i&dl=1'

            # Check if the file already exists, if not download, unpack, and delete the zip
            if not os.path.exists(self.model_save_path):
                os.makedirs(self.model_save_path, exist_ok=True)

            # Use requests to download the file
            zip_file_path = os.path.join(self.model_save_path, 'dropbox_folder.zip')
            response = requests.get(self.model_files)

            # Save the downloaded file to the path
            with open(zip_file_path, 'wb') as f:
                f.write(response.content)

            # Unzip the downloaded file
            with zipfile.ZipFile(zip_file_path, 'r') as zip_ref:
                zip_ref.extractall(self.model_save_path)

            # Remove the zip file after extraction
            os.remove(zip_file_path)
        else:
            # Running in google colab
            self.model_save_path = '/content/hours_model'
            self.model_files = 'https://www.dropbox.com/scl/fo/l8fqrr5uolxm6028hnhqj/AHvGdiy_0sHzXg2cejjdwRo?rlkey=wz5ueckmkj12hcv4ca54wls50&st=f3l5la1i&dl=0'

            # Check if the file already exists, if not download, unpack, and delete the zip
            if not os.path.exists(self.model_save_path):
                os.makedirs(self.model_save_path, exist_ok=True)
                os.system(f"wget -O dropbox_folder.zip '{self.model_files}'")
                os.system(f"unzip dropbox_folder.zip -d {self.model_save_path}")
                os.remove("dropbox_folder.zip")


    def create_widget(self, value=''):
        with self.xmt_type_output:
            self.xmt_type_output.clear_output()  # Clear the output before creating the dropdown
            self.xmt_type = widgets.Dropdown(
                options=['LBT', 'Band 4', 'Band 5/6', 'Band 7', 'Band 8', 'Band 9/10', ''],
                value=value,
                description='XMT Type:',
                layout=widgets.Layout(width='200px')
            )
            display(self.xmt_type)
            self.xmt_type.observe(self.on_xmt_type_change, names='value')
        display(self.xmt_type_output)  # Ensure dropdown is always visible

    def on_xmt_type_change(self, change):
        self.xmt_type_value = change['new']
        self.model_selection(self.xmt_type_value)

    def model_selection(self, xmt_type):
        gear_type_to_model_file = {
            'LBT': 'LBT.pkl',
            'Band 4': 'band_4.pkl',
            'Band 5/6': 'Band_5_6.pkl',
            'Band 7': 'Band_7.pkl',
            'Band 8': 'Band 8.pkl',
            'Band 9/10': 'band_910.pkl'
        }

        self.hours_model = gear_type_to_model_file.get(xmt_type, None)
        if self.hours_model:
            self.model_path = os.path.join(self.model_save_path, self.hours_model)
            clear_output(wait=True)  # Keep the dropdown visible
            self.create_widget(xmt_type)
            if xmt_type == 'LBT':
                hours_predict = LBT(self.model_path, self.xmt_type_output)
                hours_predict.create_widgets()
            elif xmt_type == 'Band 4':
                hours_predict = Band4(self.model_path, self.xmt_type_output)
                hours_predict.create_widgets()
            elif xmt_type == 'Band 5/6':
                hours_predict = Band56(self.model_path, self.xmt_type_output)
                hours_predict.create_widgets()
            elif xmt_type == 'Band 7':
                hours_predict = Band7(self.model_path, self.xmt_type_output)
                hours_predict.create_widgets()
            elif xmt_type == 'Band 8':
                hours_predict = Band8(self.model_path, self.xmt_type_output)
                hours_predict.create_widgets()
            elif xmt_type == 'Band 9/10':
                hours_predict = Band910(self.model_path, self.xmt_type_output)
                hours_predict.create_widgets()
        else:
            print('No model selected')


class LBT:
    def __init__(self, model_load_path, xmt_type_output):
        warnings.filterwarnings('ignore')
        self.model_load_path = model_load_path
        self.model = joblib.load(self.model_load_path)
        self.xmt_type_output = xmt_type_output
        print(f'\nLBT Production predictor\n')

        # Initialize the dataframes for shifts and benches
        self.one_shift_df = pd.DataFrame({
            'BenchCount': [1, 2, 3, 4, 5, 6, 7],
            'Hours': [8, 16, 24, 32, 40, 48, 56]
        })

        self.two_shift_df = pd.DataFrame({
            'BenchCount': [1, 2, 3, 4, 5, 6, 7],
            'Hours': [16, 32, 48, 64, 80, 96, 112]
        })

        self.three_shift_df = pd.DataFrame({
            'BenchCount': [1, 2, 3, 4, 5, 6, 7],
            'Hours': [24, 48, 72, 96, 120, 144, 168]
        })

    def create_widgets(self):
        # Create the input widgets (without resetting them when clearing results)
        self.Bench_count = widgets.Dropdown(
            options=[1, 2, 3, 4, 5, 6, 7],
            value=1,
            description='Bench Count:',
            layout=widgets.Layout(width='150px')
        )

        self.Shift_count = widgets.Dropdown(
            options=[1, 2, 3],
            value=2,
            description='Shift Count:',
            layout=widgets.Layout(width='150px')
        )

        self.Month_number = widgets.Dropdown(
            options=[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12],
            value=1,
            description='Month:',
            layout=widgets.Layout(width='150px')
        )

        self.Target = widgets.IntText(
            value=1,
            description='Target:',
            layout=widgets.Layout(width='150px')
        )

        # Create the output box and buttons
        self.calculate_button = widgets.Button(description="Predict")
        self.clear_button = widgets.Button(description="Clear Results")
        self.output_box = widgets.Output()

        # Link the buttons to their respective functions
        self.calculate_button.on_click(self.on_calculate_click)
        self.clear_button.on_click(self.on_clear_click)

        # Display the widgets
        display(self.Bench_count, self.Shift_count, self.Month_number, self.Target,
                self.calculate_button, self.output_box, self.clear_button)

    def on_calculate_click(self, b):
        with self.output_box:
            self.output_box.clear_output()
            if self.Target.value not in range(1, 101):
                print('Invalid target, please enter a valid target')
                return

            # Calculate hours and predictions
            hours = self.get_hours(self.Bench_count.value, self.Shift_count.value)
            predicted_time = self.predict_time_to_repair(self.Month_number.value, self.Target.value)
            if predicted_time <= 0:
                predicted_time = hours
            prediction = math.ceil(predicted_time)
            time = math.ceil(predicted_time / hours)

            # Display the predictions
            print(f'Predicted time to repair {self.Target.value} LBT transmitters is: {prediction} hours')
            print(f'On {self.Bench_count.value} benches for {self.Shift_count.value} shifts is: {time} days')

    def on_clear_click(self, b):
        # Clear only the output box (no new widgets will be created)
        self.output_box.clear_output()

    def get_hours(self, bench_count, shift_type=2):
        if shift_type == 3:
            df = self.three_shift_df
        elif shift_type == 2:
            df = self.two_shift_df
        else:
            df = self.one_shift_df

        result = df[df['BenchCount'] == bench_count]['Hours']
        if not result.empty:
            return result.values[0]
        else:
            return "Bench count not found in the DataFrame"

    def predict_time_to_repair(self, month, count):
        input_data = pd.DataFrame({
            'month': [month],
            'count': [count]
        })

        predicted_time_to_repair = self.model.predict(input_data)
        return predicted_time_to_repair[0]

class Band4:
    def __init__(self, model_load_path, xmt_type_output):
        warnings.filterwarnings('ignore')
        self.model_load_path = model_load_path
        self.model = joblib.load(self.model_load_path)
        self.xmt_type_output = xmt_type_output
        print(f'\nBand 4 Production predictor\n')

        self.one_shift_df = pd.DataFrame({
            'BenchCount': [1, 2, 3, 4, 5, 6, 7],
            'Hours': [8, 16, 24, 32, 40, 48, 56]
        })

        self.two_shift_df = pd.DataFrame({
            'BenchCount': [1, 2, 3, 4, 5, 6, 7],
            'Hours': [16, 32, 48, 64, 80, 96, 112]
        })

        self.three_shift_df = pd.DataFrame({
            'BenchCount': [1, 2, 3, 4, 5, 6, 7],
            'Hours': [24, 48, 72, 96, 120, 144, 168]
        })

    def create_widgets(self):
        # Create the input widgets
        self.Bench_count = widgets.Dropdown(
            options=[1, 2, 3, 4, 5, 6, 7],
            value=1,
            description='Bench Count:',
            layout=widgets.Layout(width='150px')
        )

        self.Shift_count = widgets.Dropdown(
            options=[1, 2, 3],
            value=2,
            description='Shift Count:',
            layout=widgets.Layout(width='150px')
        )

        self.Target = widgets.IntText(
            value=1,
            description='Target:',
            layout=widgets.Layout(width='150px')
        )

        # Create the output box and buttons
        self.calculate_button = widgets.Button(description="Predict")
        self.clear_button = widgets.Button(description="Clear Results")
        self.output_box = widgets.Output()

        # Link the buttons to their respective functions
        self.calculate_button.on_click(self.on_calculate_click)
        self.clear_button.on_click(self.on_clear_click)

        # Display the widgets
        display(self.Bench_count, self.Shift_count, self.Target, self.calculate_button, self.output_box, self.clear_button)

    def on_calculate_click(self, b):
        with self.output_box:
            self.output_box.clear_output()
            if self.Target.value not in range(1, 101):
                print('Invalid target, please enter a valid target')
                return

            # Calculate hours and predictions
            hours = self.get_hours(self.Bench_count.value, self.Shift_count.value)
            predicted_time = self.predict_time_to_repair(self.Target.value)
            if predicted_time <= 0:
                predicted_time = hours
            prediction = math.ceil(predicted_time)
            time = math.ceil(predicted_time / hours)

            # Display the predictions
            print(f'Predicted time to repair {self.Target.value} Band 4 transmitters is: {prediction} hours')
            print(f'On {self.Bench_count.value} benches for {self.Shift_count.value} shifts is: {time} days')

    def on_clear_click(self, b):
        # Clear only the output box (no new widgets will be created)
        self.output_box.clear_output()

    def get_hours(self, bench_count, shift_type=2):
        if shift_type == 3:
            df = self.three_shift_df
        elif shift_type == 2:
            df = self.two_shift_df
        else:
            df = self.one_shift_df

        result = df[df['BenchCount'] == bench_count]['Hours']
        if not result.empty:
            return result.values[0]
        else:
            return "Bench count not found in the DataFrame"

    def predict_time_to_repair(self, count):
        input_data = pd.DataFrame({
            'count': [count]
        })

        predicted_time_to_repair = self.model.predict(input_data)
        return predicted_time_to_repair[0]

class Band56:
    def __init__(self, model_load_path, xmt_type_output):
        warnings.filterwarnings('ignore')
        self.model_load_path = model_load_path
        self.model = joblib.load(self.model_load_path)
        self.xmt_type_output = xmt_type_output
        print(f'\nBand 5/6 Production predictor\n')

        # Initialize the dataframes for shifts and benches
        self.one_shift_df = pd.DataFrame({
            'BenchCount': [1, 2, 3, 4, 5, 6, 7],
            'Hours': [8, 16, 24, 32, 40, 48, 56]
        })

        self.two_shift_df = pd.DataFrame({
            'BenchCount': [1, 2, 3, 4, 5, 6, 7],
            'Hours': [16, 32, 48, 64, 80, 96, 112]
        })

        self.three_shift_df = pd.DataFrame({
            'BenchCount': [1, 2, 3, 4, 5, 6, 7],
            'Hours': [24, 48, 72, 96, 120, 144, 168]
        })

    def create_widgets(self):
        # Create the input widgets
        self.Bench_count = widgets.Dropdown(
            options=[1, 2, 3, 4, 5, 6, 7],
            value=1,
            description='Bench Count:',
            layout=widgets.Layout(width='150px')
        )

        self.Shift_count = widgets.Dropdown(
            options=[1, 2, 3],
            value=2,
            description='Shift Count:',
            layout=widgets.Layout(width='150px')
        )

        self.Month_number = widgets.Dropdown(
            options=[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12],
            value=1,
            description='Month:',
            layout=widgets.Layout(width='150px')
        )

        self.Target = widgets.IntText(
            value=1,
            description='Target:',
            layout=widgets.Layout(width='150px')
        )

        # Create the output box and buttons
        self.calculate_button = widgets.Button(description="Predict")
        self.clear_button = widgets.Button(description="Clear Results")
        self.output_box = widgets.Output()

        # Link the buttons to their respective functions
        self.calculate_button.on_click(self.on_calculate_click)
        self.clear_button.on_click(self.on_clear_click)

        # Display the widgets
        display(self.Bench_count, self.Shift_count, self.Month_number, self.Target,
                self.calculate_button, self.output_box, self.clear_button)

    def on_calculate_click(self, b):
        with self.output_box:
            self.output_box.clear_output()
            if self.Target.value not in range(1, 101):
                print('Invalid target, please enter a valid target')
                return

            # Calculate hours and predictions
            hours = self.get_hours(self.Bench_count.value, self.Shift_count.value)
            predicted_time = self.predicted_time_to_repair(self.Month_number.value, self.Target.value)
            if predicted_time <= 0:
                predicted_time = hours
            prediction = math.ceil(predicted_time)
            time = math.ceil(predicted_time / hours)

            # Display the predictions
            print(f'Predicted time to repair {self.Target.value} Band 5/6 transmitters is: {prediction} hours')
            print(f'On {self.Bench_count.value} benches for {self.Shift_count.value} shifts is: {time} days')

    def on_clear_click(self, b):
        # Clear only the output box (no new widgets will be created)
        self.output_box.clear_output()

    def get_hours(self, bench_count, shift_type=2):
        if shift_type == 3:
            df = self.three_shift_df
        elif shift_type == 2:
            df = self.two_shift_df
        else:
            df = self.one_shift_df

        result = df[df['BenchCount'] == bench_count]['Hours']
        if not result.empty:
            return result.values[0]
        else:
            return "Bench count not found in the DataFrame"

    def predicted_time_to_repair(self, month, count):
        input_data = pd.DataFrame({
            'month': [month],
            'count': [count]
        })

        predicted_time_to_repair = self.model.predict(input_data)
        return predicted_time_to_repair[0]

class Band7:
    def __init__(self, model_load_path, xmt_type_output):
        warnings.filterwarnings('ignore')
        self.model_load_path = model_load_path
        self.model = joblib.load(self.model_load_path)
        self.xmt_type_output = xmt_type_output
        print(f'\nBand 7 Production predictor\n')

        # Initialize the dataframes for shifts and benches
        self.one_shift_df = pd.DataFrame({
            'BenchCount': [1, 2, 3, 4, 5, 6, 7],
            'Hours': [8, 16, 24, 32, 40, 48, 56]
        })

        self.two_shift_df = pd.DataFrame({
            'BenchCount': [1, 2, 3, 4, 5, 6, 7],
            'Hours': [16, 32, 48, 64, 80, 96, 112]
        })

        self.three_shift_df = pd.DataFrame({
            'BenchCount': [1, 2, 3, 4, 5, 6, 7],
            'Hours': [24, 48, 72, 96, 120, 144, 168]
        })

    def create_widgets(self):
        # Create the input widgets
        self.Bench_count = widgets.Dropdown(
            options=[1, 2, 3, 4, 5, 6, 7],
            value=1,
            description='Bench Count:',
            layout=widgets.Layout(width='150px')
        )

        self.Shift_count = widgets.Dropdown(
            options=[1, 2, 3],
            value=2,
            description='Shift Count:',
            layout=widgets.Layout(width='150px')
        )

        self.Month_number = widgets.Dropdown(
            options=[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12],
            value=1,
            description='Month:',
            layout=widgets.Layout(width='150px')
        )

        self.Target = widgets.IntText(
            value=1,
            description='Target:',
            layout=widgets.Layout(width='150px')
        )

        # Create the output box and buttons
        self.calculate_button = widgets.Button(description="Predict")
        self.clear_button = widgets.Button(description="Clear Results")
        self.output_box = widgets.Output()

        # Link the buttons to their respective functions
        self.calculate_button.on_click(self.on_calculate_click)
        self.clear_button.on_click(self.on_clear_click)

        # Display the widgets
        display(self.Bench_count, self.Shift_count, self.Month_number, self.Target,
                self.calculate_button, self.output_box, self.clear_button)

    def on_calculate_click(self, b):
        with self.output_box:
            self.output_box.clear_output()
            if self.Target.value not in range(1, 101):
                print('Invalid target, please enter a valid target')
                return

            # Calculate hours and predictions
            hours = self.get_hours(self.Bench_count.value, self.Shift_count.value)
            predicted_time = self.predicted_time_to_repair(self.Target.value, self.Month_number.value)
            if predicted_time <= 0:
                predicted_time = hours
            prediction = math.ceil(predicted_time)
            time = math.ceil(predicted_time / hours)

            # Display the predictions
            print(f'Predicted time to repair {self.Target.value} Band 7 transmitters is: {prediction} hours')
            print(f'On {self.Bench_count.value} benches for {self.Shift_count.value} shifts is: {time} days')

    def on_clear_click(self, b):
        # Clear only the output box (no new widgets will be created)
        self.output_box.clear_output()

    def get_hours(self, bench_count, shift_type=2):
        if shift_type == 3:
            df = self.three_shift_df
        elif shift_type == 2:
            df = self.two_shift_df
        else:
            df = self.one_shift_df

        result = df[df['BenchCount'] == bench_count]['Hours']
        if not result.empty:
            return result.values[0]
        else:
            return "Bench count not found in the DataFrame"

    def predicted_time_to_repair(self, count, month):
        input_data = pd.DataFrame({
            'count': [count],
            'month': [month]
        })

        predicted_time_to_repair = self.model.predict(input_data)
        return predicted_time_to_repair[0]

class Band8:
    def __init__(self, model_load_path, xmt_type_output):
        warnings.filterwarnings('ignore')
        self.model_load_path = model_load_path
        self.model = joblib.load(self.model_load_path)
        self.xmt_type_output = xmt_type_output
        print(f'\nBand 8 Production predictor\n')

        # Initialize the dataframes for shifts and benches
        self.one_shift_df = pd.DataFrame({
            'BenchCount': [1, 2, 3, 4, 5, 6, 7],
            'Hours': [8, 16, 24, 32, 40, 48, 56]
        })

        self.two_shift_df = pd.DataFrame({
            'BenchCount': [1, 2, 3, 4, 5, 6, 7],
            'Hours': [16, 32, 48, 64, 80, 96, 112]
        })

        self.three_shift_df = pd.DataFrame({
            'BenchCount': [1, 2, 3, 4, 5, 6, 7],
            'Hours': [24, 48, 72, 96, 120, 144, 168]
        })

    def create_widgets(self):
        # Create the input widgets
        self.Bench_count = widgets.Dropdown(
            options=[1, 2, 3, 4, 5, 6, 7],
            value=1,
            description='Bench Count:',
            layout=widgets.Layout(width='150px')
        )

        self.Shift_count = widgets.Dropdown(
            options=[1, 2, 3],
            value=2,
            description='Shift Count:',
            layout=widgets.Layout(width='150px')
        )

        self.Month_number = widgets.Dropdown(
            options=[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12],
            value=1,
            description='Month:',
            layout=widgets.Layout(width='150px')
        )

        self.Target = widgets.IntText(
            value=1,
            description='Target:',
            layout=widgets.Layout(width='150px')
        )

        # Create the output box and buttons
        self.calculate_button = widgets.Button(description="Predict")
        self.clear_button = widgets.Button(description="Clear Results")
        self.output_box = widgets.Output()

        # Link the buttons to their respective functions
        self.calculate_button.on_click(self.on_calculate_click)
        self.clear_button.on_click(self.on_clear_click)

        # Display the widgets
        display(self.Bench_count, self.Shift_count, self.Month_number, self.Target,
                self.calculate_button, self.output_box, self.clear_button)

    def on_calculate_click(self, b):
        with self.output_box:
            self.output_box.clear_output()
            if self.Target.value not in range(1, 101):
                print('Invalid target, please enter a valid target')
                return

            # Calculate hours and predictions
            hours = self.get_hours(self.Bench_count.value, self.Shift_count.value)
            predicted_time = self.predicted_time_to_repair(self.Month_number.value, self.Target.value)
            if predicted_time <= 0:
                predicted_time = hours
            prediction = math.ceil(predicted_time)
            time = math.ceil(predicted_time / hours)

            # Display the predictions
            print(f'Predicted time to repair {self.Target.value} Band 8 transmitters is: {prediction} hours')
            print(f'On {self.Bench_count.value} benches for {self.Shift_count.value} shifts is: {time} days')

    def on_clear_click(self, b):
        # Clear only the output box (no new widgets will be created)
        self.output_box.clear_output()

    def get_hours(self, bench_count, shift_type=2):
        if shift_type == 3:
            df = self.three_shift_df
        elif shift_type == 2:
            df = self.two_shift_df
        else:
            df = self.one_shift_df

        result = df[df['BenchCount'] == bench_count]['Hours']
        if not result.empty:
            return result.values[0]
        else:
            return "Bench count not found in the DataFrame"

    def predicted_time_to_repair(self, month, count):
        # Create a DataFrame with the input data
        data = {
            'month': [month],
            'count': [count]
        }
        df = pd.DataFrame(data)

        # Feature engineering: add more features based on existing ones
        df['month_count_interaction'] = df['month'] * df['count']
        df['month_squared'] = df['month'] ** 2
        df['count_squared'] = df['count'] ** 2

        # Feature selection: use engineered features
        selected_features = [
            'month',
            'count',
            'month_count_interaction',
            'month_squared',
            'count_squared'
        ]

        # Make prediction using the model
        features_df = df[selected_features]
        predicted_time_to_repair = self.model.predict(features_df)
        return predicted_time_to_repair[0]

class Band910:
    def __init__(self, model_load_path, xmt_type_output):
        warnings.filterwarnings('ignore')
        self.model_load_path = model_load_path
        self.model = joblib.load(self.model_load_path)
        self.xmt_type_output = xmt_type_output
        print(f'\nBand 9/10 Production predictor\n')

        # Initialize the dataframes for shifts and benches
        self.one_shift_df = pd.DataFrame({
            'BenchCount': [1, 2, 3, 4, 5, 6, 7],
            'Hours': [8, 16, 24, 32, 40, 48, 56]
        })

        self.two_shift_df = pd.DataFrame({
            'BenchCount': [1, 2, 3, 4, 5, 6, 7],
            'Hours': [16, 32, 48, 64, 80, 96, 112]
        })

        self.three_shift_df = pd.DataFrame({
            'BenchCount': [1, 2, 3, 4, 5, 6, 7],
            'Hours': [24, 48, 72, 96, 120, 144, 168]
        })

    def create_widgets(self):
        # Create the input widgets
        self.Bench_count = widgets.Dropdown(
            options=[1, 2, 3, 4, 5, 6, 7],
            value=1,
            description='Bench Count:',
            layout=widgets.Layout(width='150px')
        )

        self.Shift_count = widgets.Dropdown(
            options=[1, 2, 3],
            value=2,
            description='Shift Count:',
            layout=widgets.Layout(width='150px')
        )

        self.Month_number = widgets.Dropdown(
            options=[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12],
            value=1,
            description='Month:',
            layout=widgets.Layout(width='150px')
        )

        self.Target = widgets.IntText(
            value=1,
            description='Target:',
            layout=widgets.Layout(width='150px')
        )

        # Create the output box and buttons
        self.calculate_button = widgets.Button(description="Predict")
        self.clear_button = widgets.Button(description="Clear Results")
        self.output_box = widgets.Output()

        # Link the buttons to their respective functions
        self.calculate_button.on_click(self.on_calculate_click)
        self.clear_button.on_click(self.on_clear_click)

        # Display the widgets
        display(self.Bench_count, self.Shift_count, self.Month_number, self.Target,
                self.calculate_button, self.output_box, self.clear_button)

    def on_calculate_click(self, b):
        with self.output_box:
            self.output_box.clear_output()
            if self.Target.value not in range(1, 101):
                print('Invalid target, please enter a valid target')
                return

            # Calculate hours and predictions
            hours = self.get_hours(self.Bench_count.value, self.Shift_count.value)
            predicted_time = self.predicted_time_to_repair(self.Month_number.value, self.Target.value)
            if predicted_time <= 0:
                predicted_time = hours
            prediction = math.ceil(predicted_time)
            time = math.ceil(predicted_time / hours)

            # Display the predictions
            print(f'Predicted time to repair {self.Target.value} Band 9/10 transmitters is: {prediction} hours')
            print(f'On {self.Bench_count.value} benches for {self.Shift_count.value} shifts is: {time} days')

    def on_clear_click(self, b):
        # Clear only the output box (no new widgets will be created)
        self.output_box.clear_output()

    def get_hours(self, bench_count, shift_type=2):
        if shift_type == 3:
            df = self.three_shift_df
        elif shift_type == 2:
            df = self.two_shift_df
        else:
            df = self.one_shift_df

        result = df[df['BenchCount'] == bench_count]['Hours']
        if not result.empty:
            return result.values[0]
        else:
            return "Bench count not found in the DataFrame"

    def predicted_time_to_repair(self, month, count):
        # Create a DataFrame with the input data
        data = {
            'month': [month],
            'count': [count]
        }
        df = pd.DataFrame(data)

        # Feature engineering: add more features based on existing ones
        df['month_count_interaction'] = df['month'] * df['count']
        df['month_squared'] = df['month'] ** 2
        df['count_squared'] = df['count'] ** 2

        # Feature selection: use engineered features
        selected_features = [
            'month',
            'count',
            'month_count_interaction',
            'month_squared',
            'count_squared'
        ]

        # Make prediction using the model
        features_df = df[selected_features]
        predicted_time_to_repair = self.model.predict(features_df)
        return predicted_time_to_repair[0]


# Instantiate the class and use it
hours_predict = Setup()


#<big>**$\color{tan}{\text{III. Manpower, Throughput,}}$**<br>$\color{tan}{\text{And Muster Tools}}$</big>

---



##$\color{gray}{\text{A. Manpower Throughput}}$

In [ ]:
# @markdown
import pandas as pd
import joblib
import numpy as np
import os
import requests
import zipfile
import ipywidgets as widgets
from IPython.display import display, clear_output

class setup_files:
    def __init__(self):
        self.setup_files()
        self.create_widgets()

    def setup_files(self):
        # Determine if the code is running in google colab or local runtime
        if os.path.exists(r'C:\Users\ricks\Desktop\colab'):
            self.model_save_path = r'C:\Users\ricks\Desktop\colab\capacity_model'
            self.model_files = 'https://www.dropbox.com/scl/fo/62ae5zmg66curs07v61xd/AJY6USM0nKb6sW_8kxLnILg?rlkey=z7g4uq0p1qz4ld01fvdf2utn6&st=bkqzdvlm&dl=1'
            self.model_file_name = 'capacity_local.pkl'

            # check if the file already exists, if not download, unpack, and delete the zip
            if not os.path.exists(self.model_save_path):
                os.makedirs(self.model_save_path, exist_ok=True)

            # Use requests to download the file
            zip_file_path = os.path.join(self.model_save_path, 'dropbox_folder.zip')
            response = requests.get(self.model_files)

            # Save the downloaded file to the path
            with open(zip_file_path, 'wb') as f:
                f.write(response.content)

            # Unzip the file
            with zipfile.ZipFile(zip_file_path, 'r') as zip_ref:
                zip_ref.extractall(self.model_save_path)

            # Remove the zip file after extraction
            os.remove(zip_file_path)
        else:
            self.model_save_path = '/content/capacity_model'
            self.model_files = 'https://www.dropbox.com/scl/fo/62ae5zmg66curs07v61xd/AJY6USM0nKb6sW_8kxLnILg?rlkey=z7g4uq0p1qz4ld01fvdf2utn6&st=bkqzdvlm&dl=0'
            self.model_file_name = 'capacity.pkl'
            # Check if the file already exists, if not download, unpack, and delete the zip
            if not os.path.exists(self.model_save_path):
                os.makedirs(self.model_save_path, exist_ok=True)
                os.system(f"wget -O dropbox_folder.zip '{self.model_files}'")
                os.system(f"unzip dropbox_folder.zip -d {self.model_save_path}")
                os.remove("dropbox_folder.zip")

    def create_widgets(self):
        self.sailor_input = widgets.IntText(
            value=1,
            description='Avg Available',
            layout=widgets.Layout(width='150px'),
            style={'description_width': 'initial'}
        )
        self.calculate_button = widgets.Button(description='Calculate')
        self.output_box = widgets.Output()
        self.clear_button = widgets.Button(description='Clear Results')

        # Link the button click to the function
        self.calculate_button.on_click(self.on_calculate_button_click)
        self.clear_button.on_click(self.on_clear_button_click)

        # Display the widgets
        display(self.sailor_input, self.calculate_button, self.output_box, self.clear_button)

    def on_calculate_button_click(self, b):
        with self.output_box:
            clear_output()
            try:
                # Load the saved model
                model_file_path = os.path.join(self.model_save_path, self.model_file_name)
                model = joblib.load(model_file_path)

                # Get user input for average manpower
                worker_avg = int(self.sailor_input.value)

                # Reshape the input to match the model's expected input format and provide feature name
                X = pd.DataFrame([[worker_avg]], columns=['worker_avg'])

                # Make predictions
                y_pred = model.predict(X)

                # Define the evaluation metrics
                r_value = 0.89
                r_squared = 0.80
                rmse = 4.99

                # Calculate predicted range
                r = y_pred[0]
                r1 = f'{r-rmse:.0f}'
                r2 = f'{r+rmse:.0f}'

                # Display the prediction
                print(f"Predicted RFI total: ~{r1}-{r2}\n")
                # R, R², and RMSE values for reference

               # print(f'Model evaluation metrics:')
                #print(f'R: {r_value:.2f}')
                #print(f'R²: {r_squared:.2f}')
                #print(f'RMSE: {rmse:.2f}')
            except Exception as e:
                print(f"An error occurred: {str(e)}")
                print(f'Numpy version: {np.__version__}')
                print(f'Joblib version: {joblib.__version__}')

    def on_clear_button_click(self, b):
        with self.output_box:
            clear_output()

# Instantiate the setup_files class to run the application
app = setup_files()

##$\color{gray}{\text{B. Muster}}$

In [ ]:
# @markdown
import io
import sys
import os
import zipfile
import pandas as pd
import re
import shutil
from IPython.display import Javascript
from datetime import datetime
import torch
from transformers import BertTokenizer, BertForSequenceClassification
from torch.utils.data import DataLoader, TensorDataset
import json
import warnings
import numpy as np
import requests


input(f"Please read this carefully!\n\nAll folders or multiple files must be zipped prior to upload.\nThe maximum is one year, you can not mix year time frames.\n\nIf you are ready to proceed press enter.")

pd.options.mode.copy_on_write = True

class MusterFileCleaner:
    def __init__(self, input_file, destination_root, user_year):
        self.input_file = os.path.abspath(input_file)
        self.destination_root = os.path.abspath(destination_root)
        self.user_year = user_year
        self.corrected_file_name = re.compile(
            r'\s*(corrected copy ?\d*(?:\.\d+)?|copy of|\(corrected\)|corrected|copy|\(\)|'
            r'\(corrected ?\d*\)|updated|correction)\s*|\s*frcnw\s*|\s*600\s*|\s*div\s*|'
            r'\s*muster\s*|\s*sheet\s*|\s+', re.IGNORECASE
        )

    def find_header_row(self, df):
        for idx, row in df.iterrows():
            if any('W/C' in str(cell) or 'RATE' in str(cell) or 'NAME' in str(cell) for cell in row):
                return idx
        return None

    def extract_date_from_filename(self, filename):
        """
        Extracts the day and month from the filename using regular expressions and combines
        it with the user-provided year.
        """
        # Regex pattern to capture day and month
        date_pattern = re.compile(r'(\d{1,2})([A-Za-z]{3})')
        match = date_pattern.search(filename)

        if match:
            day = match.group(1)
            month = match.group(2).upper()

            # Use the user-provided year
            year = self.user_year

            # Normalize date to YYYY-MM-DD format
            date_str = f"{day} {month} {year}"
            parsed_date = datetime.strptime(date_str, "%d %b %Y").strftime("%Y-%m-%d")

            return parsed_date
        else:
            return None

    def clean_muster_report_single_file(self):
        try:
            if self.input_file.startswith('~$'):
                print(f"Skipping temporary file: {self.input_file}")
                return
            if not self.input_file.lower().endswith('.xlsx'):
                print(f"Skipping non-xlsx file: {self.input_file}")
                return

            dirty_df = pd.read_excel(self.input_file, header=None, engine='openpyxl')
            header_row = self.find_header_row(dirty_df)

            if header_row is not None:
                cleaned_dirty_df = pd.read_excel(self.input_file, header=header_row, engine='openpyxl')
            else:
                print(f"Header row not found in file {self.input_file}.")
                return

            columns_to_check = ["PRESENT/ABSENT (IF ABSENT, STATE REASONING)", "REASON"]
            for col in columns_to_check:
                if col in cleaned_dirty_df.columns:
                    status_column = col
                    break
            else:
                print("Neither 'PRESENT/ABSENT (IF ABSENT, STATE REASONING)' nor 'REASON' columns found.")
                return

            cleaned_dirty_df = cleaned_dirty_df[["W/C", "RATE", "NAME", status_column]]
            cleaned_dirty_df.rename(columns={status_column: "STATUS"}, inplace=True)
            cleaned_dirty_df.columns = cleaned_dirty_df.columns.str.lower()
            cleaned_dirty_df["w/c"] = cleaned_dirty_df["w/c"].astype(str)

            valid_wc_values = ["64B"]
            if len(cleaned_dirty_df[cleaned_dirty_df["w/c"].isin(valid_wc_values)]) == 0:
                return

            #additional filtering functionality remove comment through ABSENT filtering for work center filtering
            """
            cleaned_dirty_df = cleaned_dirty_df[cleaned_dirty_df["W/C"].isin(valid_wc_values)]

            # Check if the filtered DataFrame is empty; if so, skip this file
            if len(cleaned_dirty_df) == 0:
                print(f"No valid 'W/C' values found in {input_file}. Skipping this file.")
                return

            # Further filter out rows with "ABSENT" in the status
            cleaned_dirty_df = cleaned_dirty_df[~cleaned_dirty_df["STATUS"].str.contains("ABSENT", case=False, na=False)]
            """

            first_column_name = cleaned_dirty_df.columns[0]
            nan_string_indices = cleaned_dirty_df[first_column_name].str.contains('nan', case=False, na=False)

            if nan_string_indices.any():
                first_nan_index = nan_string_indices.idxmax()
                cleaned_dirty_df = cleaned_dirty_df.iloc[:first_nan_index]

            cleaned_file_name = re.sub(self.corrected_file_name, '', os.path.basename(self.input_file)).strip()
            extracted_date = self.extract_date_from_filename(cleaned_file_name)

            # Append the extracted date to the DataFrame as a new column
            if extracted_date:
                cleaned_dirty_df["date"] = extracted_date
            else:
                cleaned_dirty_df["date"] = pd.NaT  # Use NaT (Not a Time) for missing dates

            # Now, explicitly convert the "Date" column to the datetime type
            cleaned_dirty_df["date"] = pd.to_datetime(cleaned_dirty_df["date"], errors='coerce')

            output_folder = self.destination_root
            if not os.path.exists(output_folder):
                os.makedirs(output_folder)
            output_file = os.path.join(output_folder, f"{cleaned_file_name}")
            cleaned_dirty_df.to_excel(output_file, index=False)
            print(f"Cleaned file saved at: {output_file}")

        except zipfile.BadZipFile:
            print(f"BadZipFile error: {self.input_file} is not a valid Excel file.")
        except ValueError as ve:
            print(f"ValueError while processing {self.input_file}: {ve}")
        except Exception as e:
            print(f"Error while processing {self.input_file}: {e}")

# Class to handle folder processing
class MusterCleaner:
    def __init__(self, main_folder_path, destination_path, user_year):
        self.main_folder_path = main_folder_path
        self.destination_path = destination_path
        self.user_year = user_year

        # Compile regex for cleaning filenames
        self.corrected_file_name = re.compile(
            r'\s*(corrected copy ?\d*(?:\.\d+)?|copy of|\(corrected\)|corrected|'
            r'copy|\(\)|\(corrected ?\d*\)|updated|correction)\s*|\s*frcnw\s*|'
            r'\s*600\s*|\s*div\s*|\s*muster\s*|\s*sheet\s*|\s+',
            re.IGNORECASE
        )

    def find_header_row(self, df):
        for idx, row in df.iterrows():
            if any('W/C' in str(cell) or 'RATE' in str(cell) or 'NAME' in str(cell) for cell in row):
                return idx
        return None

    def extract_date_from_filename(self, filename):
        """
        Extracts the day and month from the filename using regular expressions and combines
        it with the user-provided year.
        """
        # Regex pattern to capture day and month
        date_pattern = re.compile(r'(\d{1,2})([A-Za-z]{3})')
        match = date_pattern.search(filename)

        if match:
            day = match.group(1)
            month = match.group(2).upper()

            # Use the user-provided year
            year = self.user_year

            # Normalize date to YYYY-MM-DD format
            date_str = f"{day} {month} {year}"
            parsed_date = datetime.strptime(date_str, "%d %b %Y").strftime("%Y-%m-%d")

            return parsed_date
        else:
            return None

    def clean_muster_report(self, input_file, output_file, extracted_date):
        try:
            # Skip files starting with '~' or non-xlsx files
            if input_file.startswith('~$'):
                print(f"Skipping temporary file: {input_file}")
                return
            if not input_file.lower().endswith('.xlsx'):
                print(f"Skipping non-xlsx file: {input_file}")
                return

            dirty_df = pd.read_excel(input_file, header=None, engine='openpyxl')
            header_row = self.find_header_row(dirty_df)

            if header_row is not None:
                cleaned_dirty_df = pd.read_excel(input_file, header=header_row, engine='openpyxl')
            else:
                print(f"Header row not found in file {input_file}.")
                return

            columns_to_check = ["PRESENT/ABSENT (IF ABSENT, STATE REASONING)", "REASON"]

            for col in columns_to_check:
                if col in cleaned_dirty_df.columns:
                    status_column = col
                    break
            else:
                print("Neither 'PRESENT/ABSENT (IF ABSENT, STATE REASONING)' nor 'REASON' columns found.")
                return

            cleaned_dirty_df = cleaned_dirty_df[["W/C", "RATE", "NAME", status_column]]
            cleaned_dirty_df.rename(columns={status_column: "STATUS"}, inplace=True)
            cleaned_dirty_df.columns = cleaned_dirty_df.columns.str.lower()
            cleaned_dirty_df["w/c"] = cleaned_dirty_df["w/c"].astype(str)

            valid_wc_values = ["64B"]
            if len(cleaned_dirty_df[cleaned_dirty_df["w/c"].isin(valid_wc_values)]) == 0:
                return

            #additional filtering functionality remove comment through ABSENT filtering for work center filtering
            """
            cleaned_dirty_df = cleaned_dirty_df[cleaned_dirty_df["W/C"].isin(valid_wc_values)]

            # Check if the filtered DataFrame is empty; if so, skip this file
            if len(cleaned_dirty_df) == 0:
                print(f"No valid 'W/C' values found in {input_file}. Skipping this file.")
                return

            # Further filter out rows with "ABSENT" in the status
            cleaned_dirty_df = cleaned_dirty_df[~cleaned_dirty_df["STATUS"].str.contains("ABSENT", case=False, na=False)]
            """

            first_column_name = cleaned_dirty_df.columns[0]
            nan_string_indices = cleaned_dirty_df[first_column_name].str.contains('nan', case=False, na=False)

            if nan_string_indices.any():
                first_nan_index = nan_string_indices.idxmax()
                cleaned_dirty_df = cleaned_dirty_df.iloc[:first_nan_index]

            # Append the extracted date to the DataFrame as a new column
            if extracted_date:
                cleaned_dirty_df["date"] = extracted_date
            else:
                cleaned_dirty_df["date"] = pd.NaT  # Use NaT (Not a Time) for missing dates

            # Now, explicitly convert the "Date" column to the datetime type
            cleaned_dirty_df["date"] = pd.to_datetime(cleaned_dirty_df["date"], errors='coerce')

            cleaned_dirty_df.to_excel(output_file, index=False)
            row_count = len(cleaned_dirty_df)

        except zipfile.BadZipFile:
            print(f"BadZipFile error: {input_file} is not a valid Excel file.")
        except ValueError as ve:
            print(f"ValueError while processing {input_file}: {ve}")
        except Exception as e:
            print(f"Error while processing {input_file}: {e}")

    def get_latest_corrected_files(self, files):
        file_dict = {}
        corrected_pattern = re.compile(r'(CORRECTED|COPY OF|COPY|\(CORRECTION\)|CORRECTION|UPDATED|of|600)', re.IGNORECASE)

        for file in files:
            normalized_file = re.sub(r'[^a-zA-Z0-9]', '', file)
            date_part = re.sub(r'(CORRECTED|COPY OF|COPY|\(CORRECTION\)|CORRECTION|UPDATED|of|xlsx|FRCNW|600|DIV|MUSTER|SHEET)', '', normalized_file, flags=re.IGNORECASE).strip()[:7]
            cleaned_file = corrected_pattern.sub('', normalized_file).strip()
            version_num = max([int(v) for v in re.findall(r'(?:CORRECTED|COPY|CORRECTION|COPY OF|UPDATED)[ ]*(\d+)', file, re.IGNORECASE)] or [0])
            base_name = 'FRCNW600DIVMUSTERSHEETxlsx'
            full_name_key = f"{base_name} {date_part}".strip()

            if full_name_key not in file_dict or version_num > file_dict[full_name_key]['version']:
                file_dict[full_name_key] = {'file': file, 'version': version_num}

        return [entry['file'] for entry in file_dict.values()]

    def process_all_files_in_main_folder(self):
        for root, dirs, files in os.walk(self.main_folder_path):
            latest_files = self.get_latest_corrected_files(files)
            for filename in latest_files:
                if filename.endswith(".xlsx"):
                    input_file = os.path.join(root, filename)
                    relative_path = os.path.relpath(root, self.main_folder_path)

                    output_subfolder = os.path.join(self.destination_path, relative_path)
                    if not os.path.exists(output_subfolder):
                        os.makedirs(output_subfolder)

                    clean_file_name = re.sub(self.corrected_file_name, '', filename).strip()
                    output_file = os.path.join(output_subfolder, f"{clean_file_name}")

                    # Extract the date from the cleaned file name (using user-provided year)
                    extracted_date = self.extract_date_from_filename(clean_file_name)

                    self.clean_muster_report(input_file, output_file, extracted_date)

# InputHandler class for handling file or folder input
class InputHandler:
    def __init__(self, user_input, destination_root, user_year=None):
        self.user_input = user_input  # This will be the exact name from the uploaded file
        self.destination_root = destination_root
        self.user_year = user_year

    def unzip_folder(self, zip_path):
        extract_path = "/tmp/unzipped_folder"
        if not os.path.exists(extract_path):
            os.makedirs(extract_path)
        with zipfile.ZipFile(zip_path, 'r') as zip_ref:
            zip_ref.extractall(extract_path)
        return extract_path

    def delete_uploaded_file(self, file_path):
        """Delete the uploaded file after processing."""
        if os.path.exists(file_path):
            os.remove(file_path)

    def delete_tmp_folder(self, folder_path):
        """Delete the temporary folder where the zip was unpacked."""
        if os.path.exists(folder_path):
            shutil.rmtree(folder_path)

    def clean_civ_aus_from_file(self, file):
        filtered_df = file[~file['rate'].isin(['CIV', 'AUS'])]
        return filtered_df

    def combine_cleaned_files(self):
        """Combine all cleaned files from the destination directory."""
        combined_df = pd.DataFrame()

        for root, dirs, files in os.walk(self.destination_root):
            for filename in files:
                if filename.endswith(".xlsx"):
                    file_path = os.path.join(root, filename)
                    df = pd.read_excel(file_path)
                    combined_df = pd.concat([combined_df, df], ignore_index=True)

        # Save the combined DataFrame to the main /content directory
        if os.path.exists(r'c:\users\ricks\desktop\colab'):
            combined_file_path = os.path.join(r'c:\users\ricks\desktop\colab', 'combined_cleaned_muster.xlsx')
        else:
            combined_file_path = os.path.join('/content', 'combined_cleaned_muster.xlsx')
            #combined_df = self.clean_civ_aus_from_file(combined_df)
        combined_df.to_excel(combined_file_path, index=False)
        if os.path.exists('clean_muster'):
            shutil.rmtree('clean_muster')

    def handle(self):
        # Use the exact file name from the upload
        if os.path.exists(r'c:\users\ricks\desktop\colab'):
            file_path = os.path.join(r'c:\users\ricks\desktop\colab', self.user_input)
        else:
            file_path = os.path.join('/content', self.user_input)

        if os.path.exists(file_path) and file_path.lower().endswith('.zip'):
            unzipped_folder_path = self.unzip_folder(file_path)
            folder_cleaner = MusterCleaner(unzipped_folder_path, self.destination_root, self.user_year)
            folder_cleaner.process_all_files_in_main_folder()

            # Delete the temporary unzipped folder after processing
            self.delete_tmp_folder(unzipped_folder_path)

        elif os.path.isfile(file_path):
            file_cleaner = MusterFileCleaner(file_path, self.destination_root, self.user_year)
            file_cleaner.clean_muster_report_single_file()

        elif os.path.isdir(file_path):
            folder_cleaner = MusterCleaner(file_path, self.destination_root, self.user_year)
            folder_cleaner.process_all_files_in_main_folder()

        else:
            print(f"{file_path} is neither a file, a zip file, nor a folder!")

        # After processing all files, combine the cleaned files
        self.combine_cleaned_files()

        # Delete the uploaded file after processing
        self.delete_uploaded_file(file_path)
        print(f'\nProcessing Complete. Running categorizer')

if __name__ == "__main__":
    def upload_file_silently():
        if os.path.exists('/content'):
            from google.colab import files

            # Redirect output to suppress files.upload() output
            original_stdout = sys.stdout  # Save the original stdout
            try:
                sys.stdout = io.StringIO()  # Redirect stdout to a dummy stream
                uploaded = files.upload()   # Use files.upload() for Google Colab
            finally:
                sys.stdout = original_stdout  # Reset stdout no matter what
            return uploaded

        else:
             file_path = input("Enter the path to the file you want to upload: ")
             file_path = os.path.normpath(file_path)
        return file_path

    # User input for the year (use input() if running interactively)
    user_year = input(f"\nPlease enter the 4 digit year for the files: ")

    # Upload the file first
    uploaded = upload_file_silently()

    if isinstance(uploaded, dict):
        # In Google Colab, `uploaded` is a dictionary, so we can use `.keys()`
        for file_name in uploaded.keys():
            if os.path.exists(r'c:\users\ricks\desktop\colab'):
                destination_root = r'c:\users\ricks\desktop\colab\clean_muster'
        else:
            destination_root = '/content/clean_muster'


    else:
        # In local runtime, `uploaded` is a string representing the file path
        file_name = uploaded
        if os.path.exists(r'c:\users\ricks\desktop\colab'):
            destination_root = r'c:\users\ricks\desktop\colab\clean_muster'
        else:
            destination_root = '/content/clean_muster'

    # Create the InputHandler instance and call the handle method
    input_handler = InputHandler(file_name, destination_root, user_year)
    input_handler.handle()


class MusterModel:
    def __init__(self):
        warnings.filterwarnings("ignore")

        # Set up paths and links
        if os.path.exists(r'c:\users\ricks\desktop\colab'):
            self.model_save_path = r'c:\users\ricks\desktop\colab\muster_model'
            self.muster_model = 'https://www.dropbox.com/scl/fo/w2y36ky13b1qkgjzocsbc/AM2ARGTD3hnqw3tc5XO0jzw?rlkey=oilrztywx62wveo71eiyy8oyi&st=e4opt0mu&dl=1'
        else:
            self.model_save_path = '/content/muster_model'
            self.muster_model = 'https://www.dropbox.com/scl/fo/w2y36ky13b1qkgjzocsbc/AM2ARGTD3hnqw3tc5XO0jzw?rlkey=oilrztywx62wveo71eiyy8oyi&st=e4opt0mu&dl=0'

        # Download and unpack the model if it's not already available
        if not os.path.exists(self.model_save_path):
            self.download_and_unpack_model()

        # Load model, tokenizer, and category mapping
        self.load_model_and_tokenizer()
        self.load_category_mapping()

        # Prepare the data for categorization
        self.new_data = pd.read_excel('combined_cleaned_muster.xlsx')
        self.new_texts = self.new_data['status'].astype(str)  # Ensure 'STATUS' column is string

        # Tokenize the texts and create DataLoader
        self.prepare_data()

        # Run the categorization process and save the results
        self.categorize_data()

        # Calculate and save the average personnel count
        self.calculate_average_personnel_count()

    def download_and_unpack_model(self):
        if os.path.exists(r'c:\users\ricks\desktop\colab'):
            print('Downloading required files...')
            os.makedirs(r'c:\users\ricks\desktop\colab\muster_model', exist_ok=True)
            zip_file_path = os.path.join(self.model_save_path, 'dropbox_folder.zip')
            response = requests.get(self.muster_model)
            with open(zip_file_path, 'wb') as f:
                f.write(response.content)
            with zipfile.ZipFile(zip_file_path, 'r') as zip_ref:
                zip_ref.extractall(self.model_save_path)
            os.remove(zip_file_path)  # Clean up the zip file after unpacking
        else:
            os.makedirs(self.model_save_path, exist_ok=True)
            print('Downloading required files...')
            os.makedirs(self.model_save_path, exist_ok=True)
            os.system(f"wget -O dropbox_folder.zip '{self.muster_model}'")
            print('Download complete, unpacking the files...')
            os.system(f"unzip dropbox_folder.zip -d {self.model_save_path}")
            print('Files unpacked')
            os.remove("dropbox_folder.zip")  # Clean up the zip file after unpacking

    def load_model_and_tokenizer(self):
        print('Loading model and tokenizer...')
        self.model = BertForSequenceClassification.from_pretrained(self.model_save_path)
        self.tokenizer = BertTokenizer.from_pretrained(self.model_save_path)

        # Ensure the model is on GPU if available
        self.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        self.model.to(self.device)
        self.model.eval()  # Set the model in evaluation mode

    def load_category_mapping(self):
        with open(f'{self.model_save_path}/category_mapping.json', 'r') as f:
            self.category_mapping = json.load(f)

        # Create reverse mapping from index to category
        self.index_to_category = {v: k for k, v in self.category_mapping.items()}

    def prepare_data(self):
        print('Preparing data...')
        # Tokenize the new texts
        encodings = self.tokenizer(list(self.new_texts), truncation=True, padding=True, return_tensors='pt')

        # Create a DataLoader
        dataset = TensorDataset(encodings['input_ids'], encodings['attention_mask'])
        self.dataloader = DataLoader(dataset, batch_size=32)

    def categorize_data(self):
        print('Categorizing the muster, please be patient as this may take some time...')
        predictions_list = []

        # Make batched predictions
        with torch.no_grad():
            for batch in self.dataloader:
                input_ids, attention_mask = batch
                input_ids = input_ids.to(self.device)
                attention_mask = attention_mask.to(self.device)

                # Get model predictions
                outputs = self.model(input_ids=input_ids, attention_mask=attention_mask)
                batch_predictions = torch.argmax(outputs.logits, dim=-1).cpu().numpy()  # Convert predictions to CPU and numpy

                # Collect the predictions
                predictions_list.extend(batch_predictions)

        # Convert predicted label indices back to category names
        predicted_categories = [self.index_to_category[pred] for pred in predictions_list]

        # Add predictions to the original data
        self.new_data['category'] = predicted_categories

        # Save the categorized data
        self.new_data.to_excel('categorized_muster.xlsx', index=False)
        print(f"\nPredictions saved to categorized_muster.xlsx")
        os.remove('combined_cleaned_muster.xlsx')

    def calculate_average_personnel_count(self):
        print('Calculating average personnel count by work center and category...')
        # Extract month from the date
        self.new_data['date'] = pd.to_datetime(self.new_data['date'])
        self.new_data['month'] = self.new_data['date'].dt.to_period('M')

        # Group by Work Center, Category, and month, count the number of musters
        unique_days_per_month = self.new_data.groupby('month').agg(work_days=('date', 'nunique')).reset_index()
        self.new_data['date'] = self.new_data['month'].dt.to_timestamp().dt.strftime('%m/01/%Y')
        total_musters_per_category = (
            self.new_data.groupby(['w/c', 'category','month', 'date'])
            .agg(category_count=('category', 'count'))
            .reset_index()
        )

        # Add a column for the average personnel count per work center and category per month
        avg_muster_per_category = pd.merge(total_musters_per_category, unique_days_per_month, on='month')
        avg_muster_per_category['category_avg'] = np.ceil(avg_muster_per_category['category_count'] / avg_muster_per_category['work_days'])
        # Ensure the month column is always set to the 1st of the month and formatted as MM/DD/YYYY

        # Save the average count data as a separate file
        avg_muster_per_category.to_excel('avg_category_count.xlsx', index=False)
        print(f"\nAverage personnel count saved to avg_category_count.xlsx")


# Instantiate and run the class
if __name__ == "__main__":
    muster_model = MusterModel()

#<big>**$\color{tan}{\text{IV. Cannibalization}}$**</big>
---



##$\color{gray}{\text{A. Cann Source List/Count:}}$<br>$\color{gray}{\text{910, Lbt, Ueu}}$

In [ ]:
# @markdown
import pandas as pd
import json
import os
import shutil
from google.colab import userdata, files, output
import sys
import io
import ipywidgets as widgets
from IPython.display import display

class Cannibalization:
    def __init__(self):
        # Initialize the process
        self.setup_files()
        self.create_widgets()

    def setup_files(self):
        # Save path and link for Json files
        self.jsons_save_path = '/content/jsons'
        self.jsons_files = 'https://www.dropbox.com/scl/fo/kilfabdwa54yrjoqgl0fp/AC6ohCyqjdrbTtc7O6ShnoU?rlkey=qmwf7lpmtfnoht6zwhigg1mwv&st=ir2e2v5z&dl=0'
        self.cann_files_save_path = '/content/cann_files'

        input(f'This tool requires the cann_files.zip generated by the 640 reports tool.\nOnce you have generated the zip file press enter.\nYou will be prompted to upload them.\n')
        # Check if the file already exists, if not download, unpack, and delete the zip
        if not os.path.exists(self.jsons_save_path):
            os.makedirs(self.jsons_save_path, exist_ok=True)
            os.system(f"wget -O dropbox_folder.zip '{self.jsons_files}'")
            os.system(f"unzip dropbox_folder.zip -d {self.jsons_save_path}")
            os.remove("dropbox_folder.zip")

        if not os.path.exists(self.cann_files_save_path):
            os.makedirs(self.cann_files_save_path, exist_ok=True)
            print(f'\nUpload the zipped cann files\n')
            files.upload()
            os.system(f"unzip cann_files.zip -d {self.cann_files_save_path}")
            os.remove("cann_files.zip")

    def create_widgets(self):
        # Create widget for gear type
        self.gear_type = widgets.Dropdown(
            options=['LBT', '910', 'UEU', ''],
            value='',  # Set the default value
            description='Gear Type:',
            layout=widgets.Layout(width='150px'),
            disabled=False,
        )

        self.output_box = widgets.Output()

        # Display widgets
        print(f'\nSelect the gear type from the dropdown.\n')
        display(self.gear_type)
        display(self.output_box)

        # Observe changes in the dropdown value
        self.gear_type.observe(self.on_gear_type_change, names='value')

    def on_gear_type_change(self, change):
        # Triggered when the user selects a different gear type
        self.gear_type_value = change['new']  # Track the new value selected
        self.upload_awp_list()  # Prompt for AWP list upload after selection

    def upload_awp_list(self):
        # Determine the file name based on the selected gear type
        gear_type_to_file = {
            'LBT': 'lbt_out_parts.csv',
            '910': '910_out_parts.csv',
            'UEU': 'ueu_out_parts.csv'
        }

        # Get the correct AWP file based on the selected gear type
        selected_gear_type = self.gear_type_value  # Ensure gear_type_value holds the correct value
        if selected_gear_type == '':
            return  # Exit if no gear type is selected
        self.awp_file = gear_type_to_file.get(selected_gear_type)

        # Check if the AWP file already exists
        awp_file_path = f'/content/cann_files/{self.awp_file}'  # Replace '/path/to/' with the actual directory path

        if os.path.exists(awp_file_path):
            self.awp_file = awp_file_path  # Use the existing file
            self.load_data()  # Proceed to load data

    def load_data(self):
        # Select appropriate files based on the gear type
        self.cross_ref_file = '/content/jsons/part_cross_reference_dict.json'  # Cross reference data
        if self.gear_type_value == '910':
            self.b910_ipb = '/content/jsons/910_ipb_dict.json'  # Units per assembly data
        elif self.gear_type_value == 'LBT':
            self.b910_ipb = '/content/jsons/lbt_ipb_dict.json'  # Units per assembly data
        else:
            self.b910_ipb = '/content/jsons/ueu_ipb_dict.json'  # Units per assembly data

        # Load data from files
        with open(self.b910_ipb, 'r') as ipb:
            self.ipb_dict = json.load(ipb)

        self.awp = pd.read_csv(self.awp_file)

        with open(self.cross_ref_file, 'r') as cross_file:
            self.part_cross_reference = json.load(cross_file)

        # You can add further code here to process the loaded data
        print(f"Loaded data for gear type: {self.gear_type_value}\n")
        self.run_cannibalization()

    def run_cannibalization(self):
        # Perform the cannibalization calculation and save the result
        self.cannibalization_df = self.calculate_cannibalization(self.awp, self.ipb_dict, self.part_cross_reference)
        self.cannibalization_df.to_csv("cannibalization_suggestions.csv", index=False)
        print(f"\nCannibalization suggestions saved to cannibalization_suggestions.csv")

        # Instantiate CannibalizationConsolidation and pass ipb_dict and gear_type to it
        cann_project = CannibalizationConsolidation(self.ipb_dict, self.gear_type_value)

    def calculate_cannibalization(self, awp, ipb_dict, part_cross_reference):
        cannibalization_suggestions = []

        # First, group by 'e_serno' and 'ord_pn' to sum 'out_qty'
        grouped_awp = awp.groupby(['e_serno', 'ord_pn']).agg({'out_qty': 'sum', 'nomenclature': 'first'}).reset_index()

        # Get the list of all serial numbers in AWP
        all_serial_numbers = awp['e_serno'].unique()
        total_serials_in_awp = len(all_serial_numbers)  # Total number of unique serial numbers

        # Group by ord_pn to calculate total availability for each part
        grouped_by_part = grouped_awp.groupby('ord_pn')

        for ord_pn, group in grouped_by_part:
            # Get units per assembly from the IPB data
            total_units_per_assy = ipb_dict.get(ord_pn, 0)
            num_serials_with_part = len(group)

            # Total parts on order across all serial numbers
            total_out_qty_all = group['out_qty'].sum()

            # Total parts available from unlisted serial numbers (using dynamic count from AWP)
            num_unlisted_serials = total_serials_in_awp - num_serials_with_part
            total_available_from_unlisted = num_unlisted_serials * total_units_per_assy

            # Serial numbers that have the part on order (listed in the AWP)
            listed_serials_with_part = group['e_serno'].unique()

            # Serial numbers that do not have the part on order (unlisted)
            unlisted_serials = set(all_serial_numbers) - set(listed_serials_with_part)

            # Process each serial number for this part
            for idx, row in group.iterrows():
                e_serno = row['e_serno']
                part_needed = row['out_qty']

                # Calculate total available in AWP for this serial
                total_available_in_awp = total_units_per_assy * num_serials_with_part - (total_out_qty_all - part_needed)

                # Calculate cannibalization sources (exclude the current serial number and any with available_qty <= 0)
                cannibalization_sources = []
                for _, source_row in group.iterrows():
                    if source_row['e_serno'] != e_serno:
                        available_qty = total_units_per_assy - source_row['out_qty']
                        if available_qty > 0:  # Exclude sources with available_qty <= 0
                            cannibalization_sources.append((source_row['e_serno'], available_qty))

                # Add unlisted serial numbers as cannibalization sources, assuming full availability
                for unlisted_serno in unlisted_serials:
                    cannibalization_sources.append((unlisted_serno, total_units_per_assy))

                # Count the possible cannibalization sources
                possible_cannibalizations = len(cannibalization_sources)

                # Look up cross-reference sources from the part_cross_reference
                other_possible_sources = part_cross_reference.get(ord_pn, None)

                # Prepare the row dictionary
                row_dict = {
                    'nomenclature': row['nomenclature'],
                    'e_serno': e_serno,
                    'ord_pn': ord_pn,
                    'total_needed': part_needed,
                    'possible_total_avail': total_available_in_awp + total_available_from_unlisted,
                    'cann_source_count': possible_cannibalizations,
                    'other_possible_sources': ', '.join(other_possible_sources) if other_possible_sources else ''  # Join sources or leave blank
                }

                # Add cannibalization sources as individual columns (cann_source_1, cann_source_2, etc.)
                for i, (cann_serno, available_qty) in enumerate(cannibalization_sources, start=1):
                    if available_qty > 0:
                        row_dict[f'cann_source_{i}'] = f'{cann_serno}: {available_qty}'

                cannibalization_suggestions.append(row_dict)

        return pd.DataFrame(cannibalization_suggestions)

class CannibalizationConsolidation:
    def __init__(self, ipb_dict, gear_type):
        # Initialize with the ipb_dict and gear_type passed from Cannibalization
        self.ipb_dict = ipb_dict
        self.gear_type = gear_type

        # Dynamically load the correct cann matrix based on the gear type
        self.load_cann_matrix(self.gear_type)

        # Perform the consolidation and redistribution
        self.consolidated_matrix = self.consolidate_and_redistribute_manual()

        # Count the number of serial numbers with 0 in the 'Total_parts_per_sn' column
        self.zero_count = (self.consolidated_matrix['total_parts'] == 0).sum()

        # Save the output to a new Excel file
        self.save_results('cann_complete.xlsx')

    def load_cann_matrix(self, gear_type):
        # Map gear types to the appropriate matrix files
        gear_type_to_matrix_file = {
            'LBT': 'lbt_cann_matrix.csv',
            '910': '910_cann_matrix.csv',
            'UEU': 'ueu_cann_matrix.csv'
        }

        # Select the appropriate matrix file based on the gear type
        cann_matrix_file = gear_type_to_matrix_file.get(gear_type)

        if cann_matrix_file:
            # Load the selected cann matrix
            self.cann_matrix = pd.read_csv(f'/content/cann_files/{cann_matrix_file}')
            # Convert only numeric columns to numeric types (coercing errors to NaN)
            non_numeric_cols = ['nomenclature', 'current_status', 'e_serno']
            numeric_cols = self.cann_matrix.columns.difference(non_numeric_cols)
            self.cann_matrix[numeric_cols] = self.cann_matrix[numeric_cols].apply(pd.to_numeric, errors='coerce')

            # Sort by 'Total_parts_per_sn'
            self.cann_matrix = self.cann_matrix.sort_values(by='total_parts', ascending=True).reset_index(drop=True)
            print(f'\nLoaded cann matrix for gear type: {gear_type}\n')
        else:
            print(f"Error: No cann matrix found for gear type {gear_type}")

    def consolidate_and_redistribute_manual(self):
        # Consolidate and redistribute parts column by column.
        result_matrix = self.cann_matrix.copy()
        part_columns = [col for col in result_matrix.columns if col not in ['e_serno', 'total_parts', 'e_serno', 'nomenclature', 'current_status']]

        for part_num in part_columns:
            if part_num in self.ipb_dict:
                upa = self.ipb_dict[part_num]  # Get UPA for the part number
                total_parts = result_matrix[part_num].sum()  # Total available parts

                if total_parts > 0:
                    # Start consolidating from the bottom row upwards
                    row_idx = len(result_matrix) - 1
                    remaining_parts = total_parts

                    while remaining_parts > 0 and row_idx >= 0:
                        # Assign parts to this row, up to the UPA limit
                        parts_to_assign = min(upa, remaining_parts)
                        result_matrix.at[row_idx, part_num] = parts_to_assign
                        remaining_parts -= parts_to_assign

                        # Move to the previous row
                        row_idx -= 1

                    # Set values in rows above the consolidated rows to blanks
                    result_matrix.loc[:row_idx, part_num] = pd.NA

        # Update the 'Total_parts_per_sn' column and ensure it's numeric
        result_matrix['total_parts'] = result_matrix[part_columns].sum(axis=1)

        return result_matrix

    def save_results(self, output_file):
        # Save the consolidated matrix to an Excel file.
        self.consolidated_matrix.to_excel(output_file, index=False, na_rep='')
        print(f"\nConsolidation complete. Output saved to {output_file}.")
        print(f"\nTotal possible complete serial numbers after cann: {self.zero_count}")

# Instantiate the Cannibalization class to automatically run the process
cann_project = Cannibalization()


#<big>**$\color{tan}{\text{V. Notebook Fix}}$**</big>
---



##$\color{gray}{\text{A. Only Run This Code If}}$<br>$\color{gray}{\text{Restarting Does Not Work}}$

In [ ]:
# @markdown
from IPython.display import clear_output
from google.colab import runtime
import os

!pip list --format=freeze | grep -v "^-e" | xargs pip uninstall -y
!pip install numpy pandas joblib

clear_output()
os.kill(os.getpid(), 9)


#<big>**$\color{tan}{\text{VI. Code Development}}$**</big>
---

In [ ]:
# @title Midband cann project
import pandas as pd
import json

# Load JSON files
with open('midband_pn_dict.json', 'r') as midband_pn_file:
    midband_pn_dict = json.load(midband_pn_file)

with open('midband_parts_dict.json', 'r') as midband_parts_file:
    midband_parts_dict = json.load(midband_parts_file)

with open('uoc_parts_dict.json', 'r') as uoc_parts_file:
    uoc_parts_dict = json.load(uoc_parts_file)

with open('midband_uoc_dict.json', 'r') as midband_uoc_file:
    midband_uoc_dict = json.load(midband_uoc_file)

with open('part_cross_reference_dict.json', 'r') as cross_ref_file:
    part_cross_reference = json.load(cross_ref_file)

# Load the AWP CSV file, ensuring ord_pn is treated as a string
awp = pd.read_csv("midband_out_parts.csv", dtype={'ord_pn': str})

# Function to calculate cannibalization incorporating UOC logic and original nomenclature
def calculate_cannibalization(awp, midband_pn_dict, midband_parts_dict, uoc_parts_dict, midband_uoc_dict, part_cross_reference):
    cannibalization_suggestions = []

    # Convert nomenclature to part number using midband_pn_dict.json for the end item (nomenclature)
    awp['part_number'] = awp['nomenclature'].map(midband_pn_dict)

    # Group by both ord_pn and e_serno to ensure each combination is treated individually
    grouped_awp = awp.groupby(['ord_pn', 'e_serno']).agg({
        'out_qty': 'sum',
        'nomenclature': 'first',
        'part_number': 'first'  # This refers to the nomenclature's part number (end item)
    }).reset_index()

    for idx, row in grouped_awp.iterrows():
        ord_pn = str(row['ord_pn'])  # Ensure ord_pn is treated as string in output
        e_serno = row['e_serno']
        out_qty = row['out_qty']
        nomenclature = row['nomenclature']  # Retain original nomenclature (end item)
        part_number = row['part_number']    # Part number for the nomenclature (end item)

        # Determine if the part on order (ord_pn) is common or UOC-specific
        common_qty = midband_parts_dict.get(ord_pn, 0)
        uoc_qty_dict = uoc_parts_dict.get(ord_pn, {})

        # Handle parts that are only UOC-specific
        if ord_pn in uoc_parts_dict and ord_pn not in midband_parts_dict:
            # It's a UOC-specific part, restrict sources based on UOC

            # Get the UOCs for the part (ord_pn) from uoc_parts_dict
            uocs_for_part = uoc_qty_dict.keys()  # List of UOCs for the part (ord_pn)

            # Get the usable on codes for the end item's part number (nomenclature) from midband_uoc_dict
            usable_on_codes = midband_uoc_dict.get(part_number, [])

            # Initialize the allowed_serial_numbers list
            allowed_serial_numbers = []

            # Check if the end item's usable on codes match the UOCs for the part
            for uoc in usable_on_codes:
                if uoc in uocs_for_part:
                    # If there's a match, add the corresponding serial numbers to allowed_serial_numbers
                    matched_serials = awp[awp['part_number'] == part_number]['e_serno'].unique()
                    allowed_serial_numbers.extend(matched_serials)

            # Ensure the list of allowed serial numbers is unique
            allowed_serial_numbers = list(set(allowed_serial_numbers))

            total_qty = sum(uoc_qty_dict.values())  # Sum UOC-specific quantities
        else:
            # It's either common or in both lists, treat as common
            total_qty = common_qty
            if ord_pn in uoc_parts_dict:
                total_qty += sum(uoc_qty_dict.values())  # Add UOC quantities if in both
            allowed_serial_numbers = awp['e_serno'].unique()

        # Find all serial numbers that don't have the part on order for cannibalization
        serials_with_part = awp[awp['ord_pn'] == ord_pn]['e_serno'].unique()
        available_serials = set(allowed_serial_numbers) - set(serials_with_part)

        # Cannibalization sources
        cann_sources = []
        for available_serno in available_serials:
            if total_qty > 0:
                cann_sources.append((available_serno, total_qty))

        # Cross-reference other sources from the part_cross_reference
        other_possible_sources = part_cross_reference.get(ord_pn, [])

        if (total_qty * len(serials_with_part)) - (out_qty * len(serials_with_part)) < 0:
            total_avail = 0
        else:
            total_avail = (total_qty * len(serials_with_part)) - (out_qty * len(serials_with_part))

        # Prepare the output row, including nomenclature part number (end item), ord_pn, and other fields
        row_dict = {
            'nomenclature': nomenclature,  # Include the original nomenclature (end item)
            'part_number': part_number,  # Include the nomenclature's part number (end item)
            'e_serno': e_serno,
            'ord_pn': f'{ord_pn}',  # Ensure ord_pn is treated as a string in output
            'total_needed': out_qty,
            'total_available_in_awp': total_avail,
            'total_available_from_unlisted': len(available_serials) * total_qty,
            'possible_cannibalizations': len(cann_sources),
            'other_possible_sources': ', '.join(other_possible_sources) if other_possible_sources else ''
        }

        # Add cannibalization sources (e.g., cann_source_1, cann_source_2, etc.)
        for i, (cann_serno, available_qty) in enumerate(cann_sources, start=1):
            row_dict[f'cann_source_{i}'] = f'{cann_serno}: {available_qty}'

        cannibalization_suggestions.append(row_dict)

    # Convert to DataFrame
    return pd.DataFrame(cannibalization_suggestions)

# Run the calculation
cannibalization_df = calculate_cannibalization(awp, midband_pn_dict, midband_parts_dict, uoc_parts_dict, midband_uoc_dict, part_cross_reference)

# Ensure ord_pn is treated as a string when saving the output
cannibalization_df['ord_pn'] = cannibalization_df['ord_pn'].astype(str)

# Save the result to a CSV file, ensuring the correct column order
columns_order = [
    'nomenclature', 'part_number', 'e_serno', 'ord_pn', 'total_needed',
    'total_available_in_awp', 'total_available_from_unlisted',
    'possible_cannibalizations', 'other_possible_sources'
]

# Dynamically add cann sources columns based on the number of sources
cann_sources_cols = [col for col in cannibalization_df.columns if col.startswith('cann_source_')]
columns_order.extend(cann_sources_cols)

# Save with the correct column order
cannibalization_df[columns_order].to_csv("cannibalization_suggestions.csv", index=False)

# Optionally, print the dataframe for debugging
print(cannibalization_df[columns_order])


In [ ]:
# @title Cleanup
import os
import shutil

# Path to the /content folder
content_path = '/content'

# Iterate through all the files and directories in /content
for filename in os.listdir(content_path):
    file_path = os.path.join(content_path, filename)

    try:
        # Check if it's a file or a directory and remove accordingly
        if os.path.isfile(file_path) or os.path.islink(file_path):
            os.unlink(file_path)  # Remove the file or link
        elif os.path.isdir(file_path):
            shutil.rmtree(file_path)  # Remove the directory and all its contents
    except Exception as e:
        print(f"Failed to delete {file_path}. Reason: {e}")




In [ ]:
# @title Distribution project
import pandas as pd
import numpy as np
from sklearn.tree import DecisionTreeRegressor
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import Matern, ConstantKernel as C
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.ensemble import VotingRegressor
from sklearn.metrics import r2_score, mean_squared_error
import matplotlib.pyplot as plt
import joblib

# Load your data
file_path = 'dfmodel.xlsx'
df = pd.read_excel(file_path)

# Function to remove outliers based on an adjustable IQR multiplier
def remove_outliers(df, column_name, multiplier=1.5):
    Q1 = df[column_name].quantile(0.25)
    Q3 = df[column_name].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - multiplier * IQR
    upper_bound = Q3 + multiplier * IQR
    return df[(df[column_name] >= lower_bound) & (df[column_name] <= upper_bound)]

# Target column and multipliers
target_column = 'Band 4'
multipliers = {
    'LBT': 1.2,
    'Band 4': 1000.5,
    'Band 5/6': 1.2,
    'Band 7': 1.2,
    'Band 8': 1000.5,
    'Band 9/10': 1.2
}

# Get the appropriate multiplier for the current target column
multiplier = multipliers.get(target_column, 1.5)

# Perform outlier removal twice
for _ in range(2):
    df = remove_outliers(df, target_column, multiplier=multiplier)

# Prepare the data
X = df['count'].values.reshape(-1, 1)
y = df[target_column].values

# Define the Decision Tree Regressor
dt_model = DecisionTreeRegressor(max_depth=None, min_samples_split=2, min_samples_leaf=1, random_state=42)

# Define the Gaussian Process Regressor with a Matern kernel
matern_kernel = C(1.0, (1e-3, 1e5)) * Matern(length_scale=1.0, nu=2.5)
gpr_model = Pipeline([
    ('scaler', StandardScaler()),
    ('gpr', GaussianProcessRegressor(kernel=matern_kernel, n_restarts_optimizer=20, random_state=42, alpha=1e-2))
])

# Create a Voting Regressor to ensemble the Decision Tree and Gaussian Process models
ensemble_model = VotingRegressor(estimators=[('dt', dt_model), ('gpr', gpr_model)])

# Fit the ensemble model
ensemble_model.fit(X, y)

# Make predictions
predictions = ensemble_model.predict(X)

# Evaluate the ensemble model
r_squared = r2_score(y, predictions)
rmse = mean_squared_error(y, predictions, squared=False)

# Output the evaluation metrics
print(f'Ensemble Model for {target_column}')
print(f'R-squared: {r_squared:.2f}')
print(f'RMSE: {rmse:.2f}')
print('-' * 40)

# Save the ensemble model to a file
#model_filename = f'band910_ensemble_model.pkl'
model_filename = f'{target_column}_ensemble_model.pkl'
joblib.dump(ensemble_model, model_filename)
print(f'Ensemble model saved to {model_filename}')

# Plot actual vs. predicted values
def plot_actual_vs_predicted(X, y, predictions):
    # Sort the data for a smoother line plot
    sorted_indices = np.argsort(X.flatten())
    X_sorted = X[sorted_indices]
    predictions_sorted = predictions[sorted_indices]

    # Plot actual values as dots
    plt.scatter(X, y, color='blue', label='Actual Data', alpha=0.6)
    # Plot predicted values as a line
    plt.plot(X_sorted, predictions_sorted, color='red', label='Predicted Trend', linewidth=2)
    plt.title(f'Actual vs Predicted Values for {target_column}')
    plt.xlabel('Count')
    plt.ylabel(target_column)
    plt.legend()
    plt.show()

# Call the plotting function
plot_actual_vs_predicted(X, y, predictions)

# Prediction code
def make_prediction(model_filename, count_value):
    # Load the saved ensemble model
    ensemble_model = joblib.load(model_filename)
    print(f'Model loaded from {model_filename}')

    # Reshape the input for prediction
    X_new = np.array([[count_value]])

    # Make a prediction using the loaded model
    prediction = np.round(ensemble_model.predict(X_new)[0])

    # Print the prediction result
    print(f'Predicted value for count {count_value}: {prediction:.2f}')

# Get user input for making a prediction
try:
    count_value = int(input("Enter an integer value for 'count': "))
    make_prediction(model_filename, count_value)
except ValueError:
    print("Invalid input. Please enter a valid integer for 'count'.")
